# 07 · ephys modulation timecourse: drug applications (baseline, drug, washout)

This notebook analyses the **drug timecourse recordings** (A5 / M6): a long gap-free current-clamp recording
(~20 min, 0 pA) in which drugs are washed in and out. **Tags** in the ABF mark the events, e.g. "5HT hit bath", "washout",
"SP hit bath", "Ba applied", "washout". A recording can hold **several applications**: solitary (washed out before
the next) or layered (a compound added while the previous one is still in the bath).

Input: `02_ephys_anomaly-fix/output/analysis_ready.csv`. Settings: `config.yaml`. Method: `README.md`.

| Step | What happens | Review |
|---|---|---|
| 1 | select the ready timecourse recordings | which files |
| 2 | **detect spikes over the whole recording and save a detection figure (PNG + SVG)** | tags, periods, spikes, Vm timecourse |
| 3 | features per period, changes, peak response, binned timecourse | |
| 4 | save | |

## Setup
`OVERWRITE = True` replaces existing output files. Close them in Excel first.

In [ ]:
OVERWRITE = True

from pathlib import Path

import pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 250)

from ephys_analysis.baseline import BaselineSettings
from ephys_analysis.spikes import SpikeSettings
from ephys_analysis.timecourse import TimecourseSettings, detect_all, feature_table
from ephys_analysis.utils import RunLog, check_writable, find_repo_root, load_config, repo_path, stamp, write_csv

REPO = find_repo_root()
cfg = load_config(REPO / "07_ephys_timecourse")
sp = SpikeSettings.from_config(cfg["spikes"])
tc = TimecourseSettings.from_config(cfg["timecourse"])
bs = BaselineSettings.from_config(cfg["baseline"])
out = REPO / "07_ephys_timecourse" / "output"
figures = out / "figures"
targets = [out / "spikes.csv", out / "detection.csv", out / "period_features.csv",
           out / "timecourse_summary.csv", out / "timecourse_bins.csv"]
check_writable(targets, OVERWRITE)
log = RunLog(cfg, out)
print(tc)

## Step 1 · Select the timecourse recordings
The recording-book drug columns and protocol comments are read for context (they never drive the analysis).

In [ ]:
ready = pd.read_csv(repo_path(cfg, cfg["recordings_table"]), dtype={"cell_tag": "string"})
selection = ready[ready["family"].isin(cfg["families"])].reset_index(drop=True)
book_path = repo_path(cfg, cfg["book_table"])
book = pd.read_csv(book_path) if book_path.exists() else None
display(selection[["recording_id", "cell_tag", "abf_filename", "voltage_source", "bias_current_pA", "corrections"]])

## Step 2 · Detect spikes and save a detection figure for every file
**Tags** are classified by keywords (`drug_keywords`, `washout_keywords` in config); the compound is read from the tag
text (`drug_names`), else from the recording book, else "unknown" (flagged). Every drug tag starts an **application**:
- **solitary** (previous compound washed out): baseline = its own last 100 s before the tag;
- **layered** (previous compound still in the bath): condition = all compounds (e.g. "5-HT + Ba2+"), baseline = the
  **initial** baseline of the chain; the last 100 s before the tag is reported as `pre_tag`.

**Drug period** = from **60 s after the tag, 200 s long**, cut at the next tag (the same window for every cell).
**Washout** = last 100 s before the next drug tag (or the end of the recording). The silent / tonic cut-off scales
with period length (≤ 10 spikes per 100 s).

Figure per recording: the whole recording with all tags and the periods of every application, the binned firing rate
and Vm (spikes removed) against time from the first drug tag with each application's **peak response** marked
(▼ hyperpolarisation, ▲ depolarisation), and the spikes of each period.

In [ ]:
spikes, detection = detect_all(selection, sp, tc, bs, log, figures, tuple(cfg["figure_formats"]), book)
display(detection)

In [ ]:
# Review: every detection figure. Check the tags (dashed), the periods (shaded) and the spikes.
for f in detection["figure"].dropna():
    if f:
        print(Path(f).name)
        display(Image(filename=f, width=1100))

**Checkpoint.** Check that the tags were read correctly (drug names, solitary / layered), and where each drug
period sits relative to the response. The features below are computed from exactly these spikes and periods.

## Step 3 · Features
- **Per period** (`period_features.csv`, one row per recording × application × period): the 04 baseline features.
- **Summary** (`timecourse_summary.csv`, **one row per application**): condition, solitary / layered, the period
  values, **changes** (drug − baseline, washout − baseline; layered also drug − pre-tag), the **lowest / highest point**
  after the tag (30 s smoothing) and the **peak response** (whichever is larger), highest / lowest rate, and the
  **baseline drift** (flagged above 2 mV/min). 08 makes one column set per condition from this table.
- **Timecourse** (`timecourse_bins.csv`): 10 s bins of spike count, rate and Vm, in time from the first drug tag.

In [ ]:
periods, summary, bins = feature_table(selection, spikes, tc, bs, book, log)
display(periods[["abf_filename", "application", "condition", "period", "window_start_s", "window_end_s", "firing_class", "n_spikes",
                 "firing_rate_Hz", "vm_baseline_mV", "isi_cv", "ap_threshold_mV"]].round(2))

In [ ]:
display(summary.T)

## Step 4 · Save

In [ ]:
for df, p in zip((spikes, detection, periods, summary, bins), targets):
    write_csv(stamp(df, log.run_id), p)
    print("written:", p)
log.finish(targets)